# Quick Trainer on Google Colab

Fine-tune a Hugging Face language model with [Quick Trainer](https://github.com/MParvin/quick-trainer) using LoRA and optional 4-bit quantization, then upload the result to the Hugging Face Hub.

**Before you start:**
1. Go to **Runtime → Change runtime type → GPU** (T4 or better recommended).
2. Add a Colab secret named `HF_TOKEN` with your [Hugging Face write token](https://huggingface.co/settings/tokens).
3. Edit the variables in the **configuration cell** below (base model, final model name, commit message, etc.).
4. Run all cells in order.

**Gated models:** The default base model is ungated. To use Llama or other gated models, accept the license on the model's Hugging Face page while logged in, then set `BASE_MODEL` in the configuration cell.

In [ ]:
# Step 1 — Edit your run settings here before running the rest of the notebook.

# Save training outputs to Google Drive instead of ephemeral /content storage.
# Set to True to keep checkpoints after the Colab session ends.
USE_DRIVE = False

# Hugging Face model id used as the fine-tuning starting point.
# For gated models (e.g. meta-llama/Llama-3.2-1B-Instruct), accept the license
# on huggingface.co while logged in before training.
BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Final model name on the Hugging Face Hub (format: username/model-name).
# This is where the fine-tuned weights will be uploaded.
HF_REPO_ID = "your-username/my-colab-model"

# Commit message recorded on the Hugging Face Hub when the model is uploaded.
COMMIT_MESSAGE = "Fine-tuned with Quick Trainer on Colab"

# Upload the fine-tuned model to the Hugging Face Hub after training.
HF_UPLOAD_ENABLED = True

# Create the Hugging Face repo as private when uploading.
HF_PRIVATE = False

# Training datasets (samples are concatenated). Each entry needs at least "path".
# Optional keys: split, text_field, instruction_field, response_field,
# messages_field, dataset_config, code_field, docstring_field, max_samples.
DATASETS = [
    {
        "path": "databricks/databricks-dolly-15k",
        "split": "train",
        "instruction_field": "instruction",
        "response_field": "response",
    },
    # Add more datasets, for example:
    # {
    #     "path": "imdb",
    #     "split": "train",
    #     "text_field": "text",
    # },
]

# Default sample cap per dataset for faster Colab runs.
# Set to None to use each entry's max_samples or the full dataset.
MAX_SAMPLES = 500

# Number of full passes over the training data.
NUM_TRAIN_EPOCHS = 1

# Pin to a commit SHA for reproducible Colab installs (recommended).
# Example: QUICK_TRAINER_REF = "8b6906bf471935ef9d611a4d525be7d365b575f1"
QUICK_TRAINER_REF = "master"

# Allow executing custom modeling code from the Hub (unsafe; keep False unless required).
TRUST_REMOTE_CODE = False

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Enable one via Runtime → Change runtime type → GPU, "
        "then re-run this cell."
    )

print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Install a pinned ref when possible (set QUICK_TRAINER_REF in the config cell).
!pip install -q "git+https://github.com/MParvin/quick-trainer.git@{QUICK_TRAINER_REF}"

# Colab ships an old torchao (0.10.x) that is incompatible with recent PEFT,
# which breaks the LoRA merge step. Upgrade it to a supported version.
!pip install -q -U "torchao>=0.16.0"

In [ ]:
import getpass
import os

try:
    from google.colab import userdata

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab secrets.")
except Exception:
    os.environ["HF_TOKEN"] = getpass.getpass("Hugging Face token (write access): ")
    print("HF_TOKEN set from prompt.")

from huggingface_hub import login

login(token=os.environ["HF_TOKEN"])
print("Logged in to Hugging Face Hub.")

In [ ]:
if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    OUTPUT_DIR = "/content/drive/MyDrive/quick-trainer/output"
else:
    OUTPUT_DIR = "/content/output"

print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
from pathlib import Path

import yaml

CONFIG_PATH = Path("configs/colab.yaml")
CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)

datasets = []
for entry in DATASETS:
    ds = dict(entry)
    if MAX_SAMPLES is not None and "max_samples" not in ds:
        ds["max_samples"] = MAX_SAMPLES
    datasets.append(ds)

config = {
    "base_model": BASE_MODEL,
    "datasets": datasets,
    "training": {
        "output_dir": OUTPUT_DIR,
        "num_train_epochs": NUM_TRAIN_EPOCHS,
        "per_device_train_batch_size": 2,
        "gradient_accumulation_steps": 4,
        "learning_rate": 2.0e-4,
        "max_seq_length": 512,
        "use_lora": True,
        "load_in_4bit": True,
        "bf16": True,
        "lora": {"r": 16, "lora_alpha": 32, "lora_dropout": 0.05},
    },
    "huggingface": {
        "enabled": HF_UPLOAD_ENABLED,
        "repo_id": HF_REPO_ID,
        "private": HF_PRIVATE,
        "commit_message": COMMIT_MESSAGE,
    },
    "ollama": {"enabled": False},
    "trust_remote_code": TRUST_REMOTE_CODE,
    "evaluation": {"enabled": False},
}

CONFIG_PATH.write_text(yaml.dump(config, default_flow_style=False, sort_keys=False))
print(f"Wrote config to {CONFIG_PATH}")
print(f"Base model: {BASE_MODEL}")
print(f"Datasets: {[d['path'] for d in datasets]}")
print(f"Final model: {HF_REPO_ID}")

In [ ]:
!quick-trainer validate {CONFIG_PATH}

In [ ]:
!quick-trainer train --config {CONFIG_PATH}

In [ ]:
from pathlib import Path

output_dir = Path(OUTPUT_DIR)
export_dir = output_dir / "merged"

print(f"Training output: {output_dir}")
if export_dir.exists():
    print(f"Merged weights: {export_dir}")

if HF_UPLOAD_ENABLED and not HF_REPO_ID.startswith("your-username/"):
    print(f"Hugging Face repo: https://huggingface.co/{HF_REPO_ID}")
    print(f"Commit message: {COMMIT_MESSAGE}")